# 2D PINN for Diffusion Equation with Forcing

This notebook solves:

u_t = u_xx + u_yy - e^{-t}(1 - 2π²) sin(πx) sin(πy)

Exact solution:
u(x,y,t) = e^{-t} sin(πx) sin(πy)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
class PINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, x, y, t):
        return self.net(torch.cat([x, y, t], dim=1))

model = PINN().to(device)

In [ ]:
def pde_residual(model, x, y, t):
    x.requires_grad_(True)
    y.requires_grad_(True)
    t.requires_grad_(True)

    u = model(x, y, t)

    u_t = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]

    u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]

    u_y = torch.autograd.grad(u, y, torch.ones_like(u), create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, y, torch.ones_like(u_y), create_graph=True)[0]

    forcing = torch.exp(-t) * (1 - 2 * torch.pi**2) * torch.sin(torch.pi * x) * torch.sin(torch.pi * y)

    return u_t - (u_xx + u_yy) + forcing

In [ ]:
def loss_fn(model):
    N_f = 5000
    x_f = -1 + 2 * torch.rand(N_f,1,device=device)
    y_f = -1 + 2 * torch.rand(N_f,1,device=device)
    t_f = torch.rand(N_f,1,device=device)

    loss_pde = torch.mean(pde_residual(model, x_f, y_f, t_f)**2)

    N_bc = 1000
    t_bc = torch.rand(N_bc,1,device=device)

    y_rand = -1 + 2 * torch.rand(N_bc,1,device=device)
    x_rand = -1 + 2 * torch.rand(N_bc,1,device=device)

    loss_bc = (
        torch.mean(model(-torch.ones_like(x_rand), y_rand, t_bc)**2) +
        torch.mean(model(torch.ones_like(x_rand), y_rand, t_bc)**2) +
        torch.mean(model(x_rand, -torch.ones_like(y_rand), t_bc)**2) +
        torch.mean(model(x_rand, torch.ones_like(y_rand), t_bc)**2)
    )

    N_ic = 1000
    x_ic = -1 + 2 * torch.rand(N_ic,1,device=device)
    y_ic = -1 + 2 * torch.rand(N_ic,1,device=device)
    t0 = torch.zeros_like(x_ic)

    u_true = torch.sin(torch.pi * x_ic) * torch.sin(torch.pi * y_ic)
    loss_ic = torch.mean((model(x_ic,y_ic,t0) - u_true)**2)

    return loss_pde + loss_bc + loss_ic

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(3000):
    optimizer.zero_grad()
    loss = loss_fn(model)
    loss.backward()
    optimizer.step()

    if epoch % 500 == 0:
        print(f"Epoch {epoch}: {loss.item():.4f}")

In [ ]:
def evaluate(model, t_val, N=100):
    x = torch.linspace(-1,1,N)
    y = torch.linspace(-1,1,N)
    X, Y = torch.meshgrid(x, y, indexing='ij')

    Xf = X.reshape(-1,1).to(device)
    Yf = Y.reshape(-1,1).to(device)
    Tf = torch.full_like(Xf, t_val).to(device)

    with torch.no_grad():
        U = model(Xf, Yf, Tf).cpu().numpy().reshape(N,N)

    return X.numpy(), Y.numpy(), U

In [ ]:
X, Y, U = evaluate(model, 0.5)

plt.imshow(U, extent=[-1,1,-1,1], origin='lower')
plt.title("PINN Solution t=0.5")
plt.colorbar()
plt.show()

In [ ]:
def animate_video(model, filename="pinn.mp4"):
    fig, ax = plt.subplots()

    X, Y, U = evaluate(model, 0.0)
    img = ax.imshow(U, extent=[-1,1,-1,1], origin='lower', vmin=-1, vmax=1)

    def update(frame):
        t = frame / 50
        _, _, U = evaluate(model, t)
        img.set_data(U)
        ax.set_title(f"t={t:.2f}")
        return [img]

    anim = FuncAnimation(fig, update, frames=50)
    writer = FFMpegWriter(fps=10)
    anim.save(filename, writer=writer)

animate_video(model)